# Tech Challenge Fase 3 — Diagnóstico da Base de Modelagem

## Objetivo

Validar se os dados construídos na Fase 2 possuem qualidade, volume e estrutura suficientes para o desenvolvimento de um modelo supervisionado capaz de prever se um aluno será alfabetizado ou não alfabetizado.

Nesta etapa serão avaliados:
- volume de dados;
- distribuição temporal;
- balanceamento da variável-alvo;
- cobertura territorial;
- duplicidades aluno × ano;
- população candidata ao teste temporal;
- possíveis riscos de data leakage.


## 1. Visão geral da base

Verifica volume, quantidade de alunos, cobertura temporal e territorial e balanceamento do target `alfabetizado`.


In [ ]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_aluno) AS alunos_distintos,
    COUNT(DISTINCT ano) AS qtd_anos,
    MIN(ano) AS primeiro_ano,
    MAX(ano) AS ultimo_ano,
    COUNT(DISTINCT id_municipio) AS qtd_municipios,
    COUNT(DISTINCT sigla_uf) AS qtd_ufs,
    COUNT(DISTINCT rede) AS qtd_redes,
    COUNT(DISTINCT serie) AS qtd_series,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(100 * AVG(CASE WHEN alfabetizado = TRUE THEN 1.0 WHEN alfabetizado = FALSE THEN 0.0 END), 2) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos;


## 2. Distribuição temporal do target

Avalia a proporção de alfabetizados e não alfabetizados em 2023 e 2024.


In [ ]:
%sql

SELECT
    ano,
    COUNT(*) AS total_alunos,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(100 * AVG(CASE WHEN alfabetizado = TRUE THEN 1.0 WHEN alfabetizado = FALSE THEN 0.0 END), 2) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos
GROUP BY ano
ORDER BY ano;


## 3. Duplicidade aluno × ano

Um mesmo aluno pode aparecer em anos diferentes. O problema seria encontrar mais de um registro para a mesma combinação `id_aluno + ano`.


In [ ]:
%sql

SELECT COUNT(*) AS combinacoes_duplicadas
FROM (
    SELECT ano, id_aluno, COUNT(*) AS quantidade
    FROM workspace.alfabetizacao_silver.fato_alunos
    GROUP BY ano, id_aluno
    HAVING COUNT(*) > 1
);


## 4. Validação longitudinal do `id_aluno`

A repetição de um mesmo `id_aluno` entre 2023 e 2024 não é suficiente para concluir que se trata da mesma pessoa.

Como todos os registros analisados pertencem à mesma série, é feita uma validação adicional comparando município, UF e escola dos IDs que aparecem nos dois anos.

Mudanças territoriais ou escolares em proporções extremamente altas indicariam que o identificador não deve ser interpretado como chave longitudinal persistente.


In [ ]:
%sql

WITH alunos_2023 AS (
    SELECT
        id_aluno,
        id_municipio,
        sigla_uf,
        id_escola
    FROM workspace.alfabetizacao_silver.fato_alunos
    WHERE ano = 2023
),

alunos_2024 AS (
    SELECT
        id_aluno,
        id_municipio,
        sigla_uf,
        id_escola
    FROM workspace.alfabetizacao_silver.fato_alunos
    WHERE ano = 2024
)

SELECT
    COUNT(*) AS ids_presentes_nos_dois_anos,
    SUM(CASE WHEN a23.id_municipio <> a24.id_municipio THEN 1 ELSE 0 END)
        AS ids_com_municipio_diferente,
    SUM(CASE WHEN a23.sigla_uf <> a24.sigla_uf THEN 1 ELSE 0 END)
        AS ids_com_uf_diferente,
    SUM(CASE WHEN a23.id_escola <> a24.id_escola THEN 1 ELSE 0 END)
        AS ids_com_escola_diferente,
    ROUND(
        100.0 * SUM(CASE WHEN a23.id_municipio <> a24.id_municipio THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentual_municipio_diferente,
    ROUND(
        100.0 * SUM(CASE WHEN a23.id_escola <> a24.id_escola THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentual_escola_diferente
FROM alunos_2023 a23
INNER JOIN alunos_2024 a24
    ON a23.id_aluno = a24.id_aluno;


## 5. Separação temporal para modelagem

A validação longitudinal mostrou que `id_aluno` não possui evidência suficiente para ser tratado como identificador persistente da mesma pessoa entre as edições.

Por isso, a estratégia de avaliação será baseada exclusivamente no tempo:

- **2023** → desenvolvimento, treinamento e cross-validation;
- **2024** → teste final *out-of-time*.

Essa abordagem mede a capacidade de generalização do modelo para uma edição posterior da avaliação sem depender da interpretação longitudinal do identificador técnico.


In [ ]:
%sql

SELECT
    ano,
    COUNT(*) AS registros,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(
        100 * AVG(
            CASE
                WHEN alfabetizado = TRUE THEN 1.0
                WHEN alfabetizado = FALSE THEN 0.0
            END
        ),
        2
    ) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos
WHERE ano IN (2023, 2024)
GROUP BY ano
ORDER BY ano;


## 6. Valores ausentes

Verifica a completude das principais variáveis candidatas ao modelo.


In [ ]:
%sql

SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN alfabetizado IS NULL THEN 1 ELSE 0 END) AS target_nulo,
    SUM(CASE WHEN id_municipio IS NULL THEN 1 ELSE 0 END) AS municipio_nulo,
    SUM(CASE WHEN sigla_uf IS NULL THEN 1 ELSE 0 END) AS uf_nula,
    SUM(CASE WHEN rede IS NULL THEN 1 ELSE 0 END) AS rede_nula,
    SUM(CASE WHEN serie IS NULL THEN 1 ELSE 0 END) AS serie_nula,
    SUM(CASE WHEN presenca IS NULL THEN 1 ELSE 0 END) AS presenca_nula
FROM workspace.alfabetizacao_silver.fato_alunos;


## 7. Diagnóstico inicial de data leakage

A variável `proficiencia` deve ser analisada com cuidado porque a classificação de alfabetização está diretamente associada ao desempenho na avaliação.


In [ ]:
%sql

SELECT
    alfabetizado,
    COUNT(*) AS quantidade,
    ROUND(AVG(proficiencia), 2) AS media_proficiencia,
    ROUND(MIN(proficiencia), 2) AS menor_proficiencia,
    ROUND(MAX(proficiencia), 2) AS maior_proficiencia,
    ROUND(percentile_approx(proficiencia, 0.50), 2) AS mediana_proficiencia
FROM workspace.alfabetizacao_silver.fato_alunos
WHERE alfabetizado IS NOT NULL
  AND proficiencia IS NOT NULL
GROUP BY alfabetizado
ORDER BY alfabetizado;


## Conclusão do diagnóstico

O diagnóstico confirmou condições adequadas para a construção de um modelo supervisionado:

- 3.867.999 registros individuais em 2023 e 2024;
- target aproximadamente balanceado;
- ausência de duplicidade na combinação `id_aluno + ano`;
- ampla cobertura territorial;
- ausência de nulos nas principais chaves analisadas;
- `proficiencia` identificada como variável com risco direto de *data leakage* e, portanto, excluída das features;
- `serie` sem variabilidade útil para modelagem.

### Validação do identificador

Foram encontrados 1.515.671 IDs presentes nos dois anos. Entre eles, aproximadamente **84,08%** aparecem associados a município diferente e **99,88%** a escola diferente, enquanto a UF permanece igual.

Esse padrão é incompatível com o uso seguro de `id_aluno` como identificador longitudinal da mesma pessoa e sugere reutilização ou recodificação do identificador entre edições.

### Decisão metodológica

- **2023** será utilizado para desenvolvimento, treinamento e cross-validation;
- **2024 completo** será reservado como teste temporal final *out-of-time*;
- `id_aluno` será tratado apenas como identificador técnico e não como feature preditora.

O próximo passo é enriquecer a base com variáveis socioeconômicas e construir a Gold `base_modelagem_aluno`.
